# 4.3 — Soft-prefix gap across post-training stages

**Pre-registration (written 2026-10-26, before any Experiment 3 run).** Spec: `persona_geom_mixture.md`, Experiment 3.

Question: at which post-training stage does "the base model in some context" stop describing the model?

Checkpoints: OLMo 3 base → Instruct-SFT → Instruct-DPO → Instruct RL revisions step_100, step_200, step_300 → final
Instruct. The RL revisions are the only intermediate revisions on the hub, all in the final RL stage (SFT and DPO
have none). For SFT, DPO and final I reuse the existing 2,400-answer runs (same questions, suffix, default system
message, 80 tokens, T = 1). The step revisions are sampled the same way. The rendered chat-template text is identical for
all checkpoints (checked).

Per checkpoint M_k, on the hold-out half of `data/basis_split.json`:
- self-score: log P_{M_k}(own answer | question), chat template, per token (the −0.898 convention for final Instruct);
- base + soft prefix: prefix trained on the base model to M_k's training-half answers. Main setting L = 128,
  lr = 5e-3, 8 epochs, early stopping on 20% of the training questions (notebook 3.6). Check grid L ∈ {16, 64} ×
  lr ∈ {5e-3, 2e-2}. The reported number is the best by **validation**, never by hold-out;
- optimizer control: the same L = 128 prefix trained on M_k itself; must reach within 0.05 nats/token of the
  self-score, otherwise the base number for that checkpoint is flagged as uninterpretable;
- gap_k = self-score − base + prefix score, with a question bootstrap;
- arithmetic mixture (EM over the 87 base contexts) for every checkpoint. Geometric fit A for SFT, DPO and final if the
  hidden-state storage fits in the budget.

How each outcome will be read (from the spec):

| Result | Reading |
|---|---|
| Gap ≈ 0 at SFT and growing at DPO/RL | Selection describes SFT; later stages change the dynamics |
| Gap already large at SFT | Even supervised post-training goes beyond selection in this setting |

If the error band overlaps the boundary between the two rows, I will say so rather than pick one.

My own prior (for the record): the gap is already large at SFT. Notebook 1.9 found that the distance from the best persona
mixture is 0.65 nats/token at SFT vs 0.82 at final, so most of the change happens at SFT. I expect the soft prefix
to close a similar fraction (roughly 40–50%) at each stage, so gap_SFT is 0.3–0.4 and grows by less than 0.15 through
DPO and RL. Within RL (steps 100–300 → final) I expect little change.
